# 💽 読み速度と cloudflared

Dataset 読み出し帯域から起動時間を見積る。


In [ ]:
import os
import shutil
import subprocess
import time


def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    # slug 一致を最優先。無ければ qwen を含むもの、最後に全候補。
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]


ROOT = find_root()


print("=== Dataset の配置 ===")
for name in sorted(os.listdir(ROOT))[:20]:
    p = os.path.join(ROOT, name)
    print(f"  {name:32s} {'DIR' if os.path.isdir(p) else os.path.getsize(p)}")

print("\n=== 読み速度（512MB, mmap なし）===")
p = os.path.join(ROOT, "model.safetensors")
CHUNK = 8 << 20
NEED = 512 << 20
t0 = time.time()
got = 0
with open(p, "rb") as f:
    while got < NEED:
        b = f.read(CHUNK)
        if not b:
            break
        got += len(b)
dt = time.time() - t0
print(f"  {got / 2**20:.0f} MiB in {dt:.2f}s = {got / 2**20 / dt:.0f} MiB/s")
total = 19.5 * 2**30
print(f"  19.5GiB 全体の推定: {total / (got / dt):.0f}s"
      f"（TP=2 で各 GPU が半分読むなら約 {total / 2 / (got / dt):.0f}s）")

print("\n=== cloudflared ===")
print("  which:", shutil.which("cloudflared"))
try:
    r = subprocess.run(["cloudflared", "--version"], capture_output=True,
                       text=True, timeout=60)
    print("  version rc:", r.returncode,
          ((r.stdout or "") + (r.stderr or "")).strip()[:120])
except FileNotFoundError:
    print("  cloudflared 未インストール（毎回ダウンロードが必要）")
for d in ("/usr/local/bin", "/usr/bin", "/opt"):
    if os.path.isdir(d):
        hits = [n for n in os.listdir(d) if "cloud" in n.lower()]
        if hits:
            print(f"  {d}: {hits}")

print("\n=== ディスク空き ===")
s = shutil.disk_usage("/kaggle/working")
print(f"  working: free {s.free / 2**30:.1f} GiB / total {s.total / 2**30:.1f} GiB")

print("\n=== /tmp と /kaggle/working の速度差（256MB write+read）===")
for base in ("/tmp", "/kaggle/working"):
    path = os.path.join(base, "iotest.bin")
    blob = b"0" * (32 << 20)
    t0 = time.time()
    with open(path, "wb") as f:
        for _ in range(8):
            f.write(blob)
    w = time.time() - t0
    t0 = time.time()
    with open(path, "rb") as f:
        while f.read(CHUNK):
            pass
    r2 = time.time() - t0
    os.remove(path)
    print(f"  {base}: write {256 / w:.0f} MiB/s read {256 / r2:.0f} MiB/s")
